# Order-identifiability probe — single-correlated-pair geometry
All inputs/outputs live in `MyDrive/KDD_Interactions`. Run cells in order.
Cell 3 ≈ 5–10 min on CPU; Cell 4 ≈ 5 min. Cell 5 verifies from disk (range-membership checks, seed-sensitive quantities).

Interpretation notes: `frac_poly_D*` is the exact projection fraction onto the polynomial <=2-variable span (exact for polynomial-representable targets). `frac_rbf_ridge` is a **holdout test-residual variance fraction** under an RBF pairwise basis - a flexible approximation probe of the irreducible fraction, not the exact analytic projection. `additive_index` is a control case (additive-index nonlinearity, not an irreducible 3-way mechanism) and is interpreted separately from the two genuine 3-way targets.


In [ ]:
# Cell 1 — Mount Drive and set up KDD_Interactions folder
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/KDD_Interactions'
RESULTS = os.path.join(BASE, 'results')
for d in [BASE, RESULTS,
          os.path.join(RESULTS, 'order_probe_v2'),
          os.path.join(RESULTS, 'ridge_check')]:
    os.makedirs(d, exist_ok=True)
print('Base folder ready:', BASE)


In [ ]:
# Cell 2 — Core functions and config
import numpy as np, json, time, hashlib, sys

def make_data(n, rho, seed, target):
    rng = np.random.default_rng(seed)
    x1, x2, z = rng.standard_normal((3, n))
    x3 = rho * x1 + np.sqrt(max(0.0, 1 - rho**2)) * z
    X = np.column_stack([x1, x2, x3])
    if target == "monomial":
        h = x1 * x2 * x3
    elif target == "tanh_prod":
        # zero-mean factors under independence: purely 3-way at rho=0 (exact anchor)
        h = np.tanh(x1) * np.tanh(x2) * np.tanh(x3)
    elif target == "additive_index":
        # additive-index nonlinear target; contains higher-order Taylor terms,
        # but NOT a clean irreducible 3-way interaction (control case)
        h = 1.0 / (1.0 + np.exp(-(x1 + x2 + x3)))
    else:
        raise ValueError(f"Unknown target: {target}")
    return X, h

# ---------- polynomial <=2-variable basis ----------
from itertools import product as iproduct

def monomial_exps(n_vars, D, max_active=2):
    out = []
    for combo in iproduct(range(D + 1), repeat=n_vars):
        if sum(combo) <= D and sum(1 for c in combo if c > 0) <= max_active:
            out.append(combo)
    return out

def frac_poly(X, h, D, max_active=2):
    exps = monomial_exps(3, D, max_active)
    cols = []
    for e in exps:
        col = np.ones(X.shape[0])
        for j, p in enumerate(e):
            if p > 0:
                col = col * X[:, j] ** p
        cols.append(col)
    Phi = np.column_stack(cols)
    mu = Phi.mean(0); sd = Phi.std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    const_idx = exps.index((0, 0, 0))
    Phi[:, const_idx] = 1.0
    hc = h - h.mean()
    denom = float(hc @ hc)
    if denom <= 0:
        return float("nan")
    beta, *_ = np.linalg.lstsq(Phi, hc, rcond=None)
    r = hc - Phi @ beta
    return float((r @ r) / denom)

# ---------- rich RBF <=2-variable basis, ridge + holdout ----------
CENTERS = np.linspace(-2.5, 2.5, 9)
BW = 0.75

def uni_feats(x):
    return np.column_stack([x] + [np.exp(-0.5 * ((x - c) / BW) ** 2) for c in CENTERS])

def design_rbf(X):
    n = X.shape[0]
    U = [uni_feats(X[:, j]) for j in range(3)]
    cols = [np.ones((n, 1))] + U
    for a, b in [(0, 1), (0, 2), (1, 2)]:
        cols.append((U[a][:, :, None] * U[b][:, None, :]).reshape(n, -1))
    return np.concatenate(cols, axis=1)   # 1 + 30 + 300 = 331 features

def frac_rbf_ridge(X, h, lam=10.0, split_seed=0, train_frac=0.75):
    """Holdout test-residual variance fraction under the RBF pairwise basis.

    A flexible approximation probe of the irreducible fraction (low value =
    "absorbed by this <=2-variable RBF basis"), NOT the exact analytic projection.
    Ridge does not penalize the intercept (P[0,0] = 0)."""
    n = X.shape[0]
    idx = np.random.default_rng(split_seed).permutation(n)
    tr, te = idx[: int(train_frac * n)], idx[int(train_frac * n):]
    Phi = design_rbf(X)
    mu = Phi[tr].mean(0); sd = Phi[tr].std(0); sd[sd == 0] = 1.0
    Phi = (Phi - mu) / sd
    Phi[:, 0] = 1.0
    hm = h[tr].mean()
    P = np.eye(Phi.shape[1])
    P[0, 0] = 0.0                      # do not penalize the intercept
    A = Phi[tr].T @ Phi[tr] + lam * P
    b = Phi[tr].T @ (h[tr] - hm)
    try:
        from scipy.linalg import cho_factor, cho_solve
        beta = cho_solve(cho_factor(A), b)   # A is SPD with ridge
    except Exception:
        beta = np.linalg.solve(A, b)
    resid = (h[te] - hm) - Phi[te] @ beta
    denom = np.sum((h[te] - h[te].mean()) ** 2)
    if denom <= 0:
        return float("nan")
    return float((resid @ resid) / denom)

# config
N = 60_000
SEEDS = list(range(3))
RHOS = [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
TARGETS = ["monomial", "tanh_prod", "additive_index"]
POLY_DEGS = [3, 4]
RIDGE_LAM = 10.0

CODE_SHA = hashlib.sha256(b"".join(
    f.__code__.co_code for f in
    [make_data, monomial_exps, frac_poly, uni_feats, design_rbf, frac_rbf_ridge]
)).hexdigest()
print("code sha256:", CODE_SHA)
print("numpy:", np.__version__, "| python:", sys.version.split()[0])


In [ ]:
# Cell 3 — Experiment 1: order-probe sweep (poly D=3,4 + RBF-ridge), per-seed CSV
import csv, os

EXP = "order_probe_v2"
outdir = os.path.join(RESULTS, EXP)
t0 = time.time()
rows = []
for target in TARGETS:
    for rho in RHOS:
        for s in SEEDS:
            X, h = make_data(N, rho, s, target)
            rec = {"experiment": EXP, "target": target, "rho": rho, "seed": s}
            for D in POLY_DEGS:
                rec[f"frac_poly_D{D}"] = frac_poly(X, h, D)
            rec["frac_rbf_ridge"] = frac_rbf_ridge(X, h, lam=RIDGE_LAM, split_seed=s)
            rows.append(rec)
        print(f"{target:15s} rho={rho:4.2f} done  ({time.time()-t0:6.1f}s)", flush=True)

per_seed_path = os.path.join(outdir, "per_seed.csv")
with open(per_seed_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

# summary (mean/sd over seeds)
import collections
summ = collections.defaultdict(list)
for r in rows:
    summ[(r["target"], r["rho"])].append(r)
sum_rows = []
for (target, rho), rs in summ.items():
    rec = {"experiment": EXP, "target": target, "rho": rho, "n_seeds": len(rs)}
    for col in [f"frac_poly_D{D}" for D in POLY_DEGS] + ["frac_rbf_ridge"]:
        vals = [r[col] for r in rs]
        rec[col + "_mean"] = float(np.mean(vals))
        rec[col + "_sd"] = float(np.std(vals))
    sum_rows.append(rec)
res_path = os.path.join(outdir, "results.csv")
with open(res_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(sum_rows[0].keys()))
    w.writeheader(); w.writerows(sum_rows)

with open(os.path.join(outdir, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "N": N, "seeds": SEEDS, "rhos": RHOS, "targets": TARGETS,
               "poly_degs": POLY_DEGS, "ridge_lam": RIDGE_LAM,
               "code_sha256": CODE_SHA, "numpy": np.__version__}, f, indent=2)
print("wrote:", per_seed_path)
print("wrote:", res_path)


In [ ]:
# Cell 4 — Experiment 2: ridge lambda sensitivity at the anchors
import csv, os

EXP2 = "ridge_check"
outdir2 = os.path.join(RESULTS, EXP2)
rows2 = []
t0 = time.time()
for target in ["monomial", "tanh_prod"]:
    for rho in [0.0, 0.5, 1.0]:
        for lam in [1.0, 10.0, 100.0]:
            for s in SEEDS:
                X, h = make_data(N, rho, s, target)
                rows2.append({"experiment": EXP2, "target": target, "rho": rho,
                              "lam": lam, "seed": s,
                              "frac_rbf_ridge": frac_rbf_ridge(X, h, lam=lam, split_seed=s)})
        print(f"{target:10s} rho={rho:4.2f} done  ({time.time()-t0:6.1f}s)", flush=True)

ps2 = os.path.join(outdir2, "per_seed.csv")
with open(ps2, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows2[0].keys()))
    w.writeheader(); w.writerows(rows2)
with open(os.path.join(outdir2, "metadata.json"), "w") as f:
    json.dump({"experiment": EXP2, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
               "N": N, "seeds": SEEDS, "code_sha256": CODE_SHA,
               "numpy": np.__version__}, f, indent=2)
print("wrote:", ps2)


In [ ]:
# Cell 5 — Verification: read results back FROM DISK and check anchors
import csv, os
import numpy as np

def load(path):
    with open(path) as f:
        return list(csv.DictReader(f))

rows = load(os.path.join(RESULTS, "order_probe_v2", "per_seed.csv"))
assert all(r["experiment"] == "order_probe_v2" for r in rows), "experiment stamp mismatch (stale file?)"

def agg(target, rho, col):
    v = [float(r[col]) for r in rows if r["target"] == target and abs(float(r["rho"]) - rho) < 1e-9]
    return np.mean(v), np.std(v), len(v)

print(f"{'target':15s} {'rho':>5s}  {'poly D=3':>13s}  {'poly D=4':>13s}  {'RBF ridge (holdout)':>20s}")
for target in ["monomial", "tanh_prod", "additive_index"]:
    for rho in [0.0, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]:
        cells = []
        for col in ["frac_poly_D3", "frac_poly_D4", "frac_rbf_ridge"]:
            m, sd, k = agg(target, rho, col)
            cells.append(f"{m:5.3f}±{sd:4.3f}")
        print(f"{target:15s} {rho:5.2f}  " + "  ".join(f"{c:>20s}" if i == 2 else f"{c:>13s}" for i, c in enumerate(cells)))
    print()

checks = []
m, _, _ = agg("monomial", 1.0, "frac_poly_D4");   checks.append(("monomial rho=1 -> ~0 (2-variable degeneration)", m < 0.02))
m, _, _ = agg("tanh_prod", 1.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=1 -> ~0 under RBF (basis-adequacy)", m < 0.02))
m, _, _ = agg("tanh_prod", 0.0, "frac_rbf_ridge"); checks.append(("tanh_prod rho=0 -> ~1 (zero-mean product anchor)", 0.95 < m < 1.10))
m, _, _ = agg("monomial", 0.5, "frac_poly_D4");    checks.append(("monomial rho=0.5 in [0.28,0.33] (range membership)", 0.28 < m < 0.33))
m, _, _ = agg("additive_index", 0.0, "frac_rbf_ridge"); checks.append(("additive_index control small (<0.05)", m < 0.05))

story = []
for name, ok in checks:
    line = ("PASS  " if ok else "FAIL  ") + name
    story.append(line); print(line)
with open(os.path.join(RESULTS, "order_probe_v2", "check.txt"), "w") as f:
    f.write("\n".join(story) + "\n")
print("\nwrote check.txt")
